In [1]:
import pandas as pd
import os
from sklearn.datasets import load_svmlight_file

def load_local_gas_drift(folder_path='Dataset'):
    all_batches = []
    
    # Iterate through batch1.dat to batch10.dat in your local folder
    for i in range(1, 11):
        file_path = os.path.join(folder_path, f"batch{i}.dat")
        
        if os.path.exists(file_path):
            # Load SVMLight format (common for high-dimensional UCI data)
            X_sparse, y = load_svmlight_file(file_path)
            
            # Convert to dense DataFrame
            df_batch = pd.DataFrame(X_sparse.toarray())
            
            # Add target and domain metadata
            df_batch['target_y'] = y
            df_batch['batch_id'] = i  # This represents the 'k' in your source tasks
            
            all_batches.append(df_batch)
            print(f"Loaded {file_path}: {df_batch.shape}")
        else:
            print(f"Warning: {file_path} not found.")

    if not all_batches:
        return None

    # Combine all batches
    full_df = pd.concat(all_batches, axis=0).reset_index(drop=True)
    
    # Rename features to feat_0, feat_1, ... feat_127
    feature_cols = [c for c in full_df.columns if isinstance(c, int)]
    full_df = full_df.rename(columns={c: f"feat_{c}" for c in feature_cols})

    return full_df

# Run the loader
# Ensure your 'Dataset' folder is in the same directory as this script
df = load_local_gas_drift('Dataset')

if df is not None:
    print("\nSuccessfully created DataFrame.")
    print(f"Dimensions: {df.shape[0]} rows x {df.shape[1]} columns")
    print(df[['feat_0', 'target_y', 'batch_id']].head())

Loaded Dataset\batch1.dat: (445, 130)
Loaded Dataset\batch2.dat: (1244, 130)
Loaded Dataset\batch3.dat: (1586, 130)
Loaded Dataset\batch4.dat: (161, 130)
Loaded Dataset\batch5.dat: (197, 130)
Loaded Dataset\batch6.dat: (2300, 130)
Loaded Dataset\batch7.dat: (3613, 130)
Loaded Dataset\batch8.dat: (294, 130)
Loaded Dataset\batch9.dat: (470, 130)
Loaded Dataset\batch10.dat: (3600, 130)

Successfully created DataFrame.
Dimensions: 13910 rows x 130 columns
       feat_0  target_y  batch_id
0  15596.1621       1.0         1
1  26402.0704       1.0         1
2  42103.5820       1.0         1
3  42825.9883       1.0         1
4  58151.1757       1.0         1


In [2]:
beta_index_dict = {}

In [9]:
import CoRT_builder
from sklearn.linear_model import Lasso
from helper import get_target_data, get_source_data
import warnings
import numpy as np
from configs import CONST_C

warnings.simplefilter(action='ignore', category=FutureWarning)

p = df.shape[1] - 2  
K = 5
T = 5

iteration = 100
n_target = 50
n_source = 100

for i in range(iteration):
    target_data = get_target_data(df, n_target)
    source_data = get_source_data(df, n_source)

    CoRT_model = CoRT_builder.CoRT()
    similar_source_index = CoRT_model.find_similar_source(p, n_target, K, target_data, source_data, T=T, verbose=False)
    X_combined, y_combined = CoRT_model.prepare_CoRT_data(similar_source_index, source_data, target_data)

    N = X_combined.shape[0]
    lamda = CONST_C 

    model = Lasso(alpha=lamda / N, fit_intercept=False, tol=1e-14, max_iter=1000000)  
    model.fit(X_combined, y_combined.ravel())
    beta_hat_target = model.coef_[-p:]

    M_obs = np.array([i for i, b in enumerate(beta_hat_target) if b != 0])

    if len(M_obs) == 0:
        print(f"Iteration {iter}: Lasso selected no features. Skipping.")

    print(f"M_obs = {M_obs}")

    for i in M_obs:
        if i not in beta_index_dict.keys():
            beta_index_dict[i] = 1
        else:
            beta_index_dict[i] = beta_index_dict[i] + 1

M_obs = [ 10  11  25 100]
M_obs = [  8  11  12  25  27  99 107 108]
M_obs = [  8  10  11  25  99 100]
M_obs = [  8  10  13  25  95  99 100]
M_obs = [  8  10  11  14  25  95  99 100]
M_obs = [  8  10  13  14  25  95 100]
M_obs = [ 11  13  14  25  95  99 100]
M_obs = [  8  11  99 100]
M_obs = [  8  11  12  99 100]
M_obs = [  8  10  11  25  99 100]
M_obs = [ 10  11  25  44 100 108]
M_obs = [  8  10  25  95  99 100]
M_obs = [  8  10  11  25  95  99 100]
M_obs = [  8  10  11  25  99 100]
M_obs = [  8  10  25  26 100 108]
M_obs = [ 10  25  99 100 108]
M_obs = [  8  11  17  25  95 100]
M_obs = [  8  10  17  25  95 100 108]
M_obs = [  8  10  25  36  95  99 100]
M_obs = [ 10  25  26  99 100]
M_obs = [  8  10  11  17  95  99 100]
M_obs = [  8  11  25  26  99 100 108]
M_obs = [  8  10  25  95  99 100]
M_obs = [  8  10  12  25  99 100 108]
M_obs = [  8  10  11  25  95 100]
M_obs = [ 10  11  25  99 100]
M_obs = [ 8 10 27 99]
M_obs = [  8  10  11  12  25  95  99 100]
M_obs = [  8  10  11  17  95  99

In [10]:
sorted_dict = dict(sorted(beta_index_dict.items(), key=lambda item: item[1], reverse=True))
sorted_dict

{np.int64(100): 182,
 np.int64(10): 164,
 np.int64(25): 160,
 np.int64(8): 150,
 np.int64(99): 132,
 np.int64(95): 106,
 np.int64(11): 100,
 np.int64(108): 45,
 np.int64(27): 36,
 np.int64(12): 31,
 np.int64(17): 25,
 np.int64(26): 20,
 np.int64(36): 20,
 np.int64(14): 10,
 np.int64(44): 8,
 np.int64(123): 6,
 np.int64(13): 6,
 np.int64(107): 4,
 np.int64(20): 4,
 np.int64(28): 3,
 np.int64(124): 2,
 np.int64(49): 2,
 np.int64(106): 1,
 np.int64(98): 1,
 np.int64(116): 1,
 np.int64(0): 1,
 np.int64(15): 1,
 np.int64(52): 1,
 np.int64(1): 1}

In [12]:
df.head(100)

,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,...,feat_120,feat_121,feat_122,feat_123,feat_124,feat_125,feat_126,feat_127,target_y,batch_id
0,15596.1621,1.868245,2.371604,2.803678,7.512213,-2.739388,-3.344671,-4.847512,15326.6914,1.768526,...,3037.0390,3.972203,0.527291,0.728443,1.445783,-0.545079,-0.902241,-2.654529,1.0,1
1,26402.0704,2.532401,5.411209,6.509906,7.658469,-4.722217,-5.817651,-7.518333,23855.7812,2.164706,...,4176.4453,4.281373,0.980205,1.628050,1.951172,-0.889333,-1.323505,-1.749225,1.0,1
2,42103.5820,3.454189,8.198175,10.508439,11.611003,-7.668313,-9.478675,-12.230939,37562.3008,2.840403,...,5914.6685,5.396827,1.403973,2.476956,3.039841,-1.334558,-1.993659,-2.348370,1.0,1
3,42825.9883,3.451192,12.113940,16.266853,39.910056,-7.849409,-9.689894,-11.921704,38379.0664,2.851173,...,6147.4744,5.501071,1.981933,3.569823,4.049197,-1.432205,-2.146158,-2.488957,1.0,1
4,58151.1757,4.194839,11.455096,15.715298,17.654915,-11.083364,-13.580692,-16.407848,51975.5899,3.480866,...,8158.6449,7.174334,1.993808,3.829303,4.402448,-1.930107,-2.931265,-4.088756,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,15687.6016,1.614901,4.452284,7.596702,8.649012,-3.148958,-3.687689,-5.041160,16607.1054,1.718371,...,31163.4599,24.446958,9.429338,18.829755,20.809085,-9.310764,-16.332122,-17.795932,2.0,1
96,2382.6875,1.108148,0.441225,0.620207,2.342228,-0.454608,-0.657530,-2.052153,2140.9160,1.103204,...,4148.7085,5.514478,0.671660,0.818093,1.209134,-0.769831,-1.154161,-1.533679,2.0,1
97,8100.8828,1.369479,1.990999,2.733511,3.630374,-1.621131,-2.081185,-3.495362,7744.4355,1.375848,...,11217.3602,9.966007,3.500982,5.582239,6.125912,-2.840364,-4.586015,-5.202862,2.0,1
98,11839.7793,1.542422,3.030708,4.240349,5.100687,-2.363224,-2.927259,-4.056444,11863.0918,1.584723,...,17494.7922,14.074407,6.056191,11.045209,11.994464,-4.786885,-8.228770,-8.840932,2.0,1
